<a href="https://colab.research.google.com/github/YAMO-M/South-African-History-Extractor/blob/main/SA_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

mount Drive (so progress survives disconnects):

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


install dependencies:

In [14]:
!pip install -q trafilatura

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 22.8 MB/s eta 0:00:00


In [2]:
!pip install -q requests beautifulsoup4 lxml tqdm

write the scraper script to disk

In [15]:
%%writefile sahistory_scraper.py
#!/usr/bin/env python3
"""
SAHO (South African History Online) Dataset Builder
=====================================================

Builds a structured dataset from https://sahistory.org.za using:
  - sitemap.xml for URL discovery
  - embedded Schema.org JSON-LD on each page for structured fields
  - trafilatura for full article/biography body text extraction
  - the Timeline API (/api/timeline/v2/...) for event records, which is
    far more efficient than one HTTP request per event

Respects the site's published crawl rules (robots.txt: Crawl-delay 1s for
AI crawlers) and its licensing terms (llm.txt: CC BY-NC-SA 4.0 -- keep
attribution, non-commercial use only, share derivatives under the same
license).
"""

import argparse
import csv
import html
import json
import re
import sys
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

try:
    import trafilatura
except ImportError:
    trafilatura = None  # we'll fall back to selector-based extraction

try:
    from tqdm import tqdm
except ImportError:  # tqdm is optional
    def tqdm(iterable, **kwargs):
        return iterable

BASE = "https://sahistory.org.za"
SITEMAP_URL = f"{BASE}/sitemap.xml"

HEADERS = {
    # Identify yourself honestly -- include a contact so SAHO can reach you
    # if there's ever an issue. Edit the email below.
    "User-Agent": "SAHODatasetBot/1.0 (+mailto:your-email@example.com; "
                  "educational research dataset; see https://sahistory.org.za/llm.txt)"
}

CRAWL_DELAY = 1.1  # seconds -- robots.txt specifies Crawl-delay: 1 for AI bots

# Map our dataset "type" labels to URL path prefixes used on the site.
TYPE_PATH_PREFIXES = {
    "biography": ("/people/",),
    "article": ("/article/",),
    "archive": ("/archive/",),
    "place": ("/place/", "/places/"),
}

# --------------------------------------------------------------------------
# Sitemap discovery
# --------------------------------------------------------------------------

LOC_RE = re.compile(r"<loc>(.*?)</loc>", re.IGNORECASE | re.DOTALL)


def get_all_sitemap_urls(session, sitemap_url=SITEMAP_URL, seen=None):
    if seen is None:
        seen = set()
    if sitemap_url in seen:
        return []
    seen.add(sitemap_url)

    resp = session.get(sitemap_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    text = resp.text

    locs = [html.unescape(m.group(1).strip()) for m in LOC_RE.finditer(text)]

    if "<sitemapindex" in text:
        urls = []
        for loc in locs:
            urls.extend(get_all_sitemap_urls(session, loc, seen))
        return urls
    else:
        return locs


def classify_url(url):
    for type_label, prefixes in TYPE_PATH_PREFIXES.items():
        for prefix in prefixes:
            if prefix in url:
                return type_label
    return None


# --------------------------------------------------------------------------
# Page fetch + JSON-LD extraction + full body text extraction
# --------------------------------------------------------------------------

BODY_SELECTORS = [
    ".field--name-body",
    ".field--type-text-with-summary",
    ".field--name-field-body",
    "article .field--name-body",
    "main .text-formatted",
    ".node__content",
    ".field--name-field-description",
]

MIN_USABLE_TEXT_LEN = 200


def extract_jsonld(soup):
    records = []
    for script in soup.find_all("script", {"type": "application/ld+json"}):
        if not script.string:
            continue
        try:
            data = json.loads(script.string)
        except json.JSONDecodeError:
            continue
        if isinstance(data, list):
            records.extend(data)
        elif isinstance(data, dict):
            if "@graph" in data and isinstance(data["@graph"], list):
                records.extend(data["@graph"])
            else:
                records.append(data)
    return records


def extract_full_text(html_text, jsonld_records, soup):
    """Best-effort extraction of the full article/biography body text.

    Tries, in order:
      1. An explicit "articleBody"/"text" field in JSON-LD.
      2. trafilatura's general-purpose readability extraction.
      3. A hand-picked list of common Drupal "body" field CSS selectors.
    """
    for block in jsonld_records:
        body = block.get("articleBody") or block.get("text")
        if isinstance(body, str) and len(body) > MIN_USABLE_TEXT_LEN:
            return body.strip()

    if trafilatura is not None:
        extracted = trafilatura.extract(
            html_text,
            include_comments=False,
            include_tables=True,
            favor_precision=True,
        )
        if extracted and len(extracted) > MIN_USABLE_TEXT_LEN:
            return extracted.strip()

    for selector in BODY_SELECTORS:
        el = soup.select_one(selector)
        if el:
            text = el.get_text("\n", strip=True)
            if len(text) > MIN_USABLE_TEXT_LEN:
                return text

    return None


def fetch_page_data(url, session):
    """Fetch a page once and return (jsonld_records, full_text)."""
    resp = session.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    html_text = resp.text
    soup = BeautifulSoup(html_text, "lxml")

    jsonld_records = extract_jsonld(soup)
    full_text = extract_full_text(html_text, jsonld_records, soup)

    return jsonld_records, full_text


def scrape_page_type(type_label, urls, out_path, limit, session, delay):
    """Scrape a list of page URLs of one content type into a JSONL file.

    A URL only counts as "already done" if its saved record has usable
    full_text -- so re-running against an older output file (saved before
    full-text extraction existed) automatically re-fetches those URLs to
    fill in the missing text.
    """
    out_path.parent.mkdir(parents=True, exist_ok=True)

    already_done = set()
    if out_path.exists():
        with out_path.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if rec.get("full_text"):
                    already_done.add(rec.get("_source_url"))
        print(f"[{type_label}] resuming: {len(already_done)} already saved with full text")

    todo = [u for u in urls if u not in already_done]
    if limit:
        todo = todo[: max(0, limit - len(already_done))]

    if not todo:
        print(f"[{type_label}] nothing new to fetch")
        return

    print(f"[{type_label}] fetching {len(todo)} pages...")
    with out_path.open("a", encoding="utf-8") as f:
        for url in tqdm(todo, desc=type_label):
            try:
                jsonld_records, full_text = fetch_page_data(url, session)
            except requests.RequestException as e:
                print(f"  ERROR fetching {url}: {e}", file=sys.stderr)
                time.sleep(delay)
                continue

            record = {
                "_source_url": url,
                "_type_label": type_label,
                "_fetched_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                "jsonld": jsonld_records,
                "full_text": full_text,
                "full_text_word_count": len(full_text.split()) if full_text else 0,
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()
            time.sleep(delay)


DECADES = (
    ["pre1500"]
    + [str(y) for y in range(1500, 2030, 10)]
)


def scrape_events_via_timeline_api(out_path, session, delay, limit=None):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    done_decades = set()
    count = 0
    if out_path.exists():
        with out_path.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    done_decades.add(rec.get("_decade"))
                    count += 1
                except json.JSONDecodeError:
                    continue
        print(f"[event] resuming: {count} events across {len(done_decades)} decades already saved")

    with out_path.open("a", encoding="utf-8") as f:
        for decade in tqdm(DECADES, desc="event decades"):
            if decade in done_decades:
                continue
            url = f"{BASE}/api/timeline/v2/events/{decade}"
            try:
                resp = session.get(url, headers=HEADERS, timeout=30)
                resp.raise_for_status()
                data = resp.json()
            except (requests.RequestException, ValueError) as e:
                print(f"  ERROR fetching {url}: {e}", file=sys.stderr)
                time.sleep(delay)
                continue

            events = data if isinstance(data, list) else data.get("events", data)
            if isinstance(events, dict):
                events = list(events.values())

            for ev in events:
                f.write(json.dumps(
                    {"_decade": decade, "_fetched_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), **ev},
                    ensure_ascii=False,
                ) + "\n")
                count += 1
            f.flush()
            time.sleep(delay)
            if limit and count >= limit:
                break

    print(f"[event] total events saved: {count}")


def flatten_jsonld(record):
    flat = {
        "url": record.get("_source_url"),
        "type_label": record.get("_type_label"),
        "fetched_at": record.get("_fetched_at"),
    }
    jsonld_list = record.get("jsonld") or []
    main = None
    for block in jsonld_list:
        t = block.get("@type")
        if t and t not in ("BreadcrumbList", "WebPage", "WebSite", "Organization"):
            main = block
            break
    if main is None and jsonld_list:
        main = jsonld_list[0]
    main = main or {}

    flat["schema_type"] = main.get("@type")
    flat["name"] = main.get("name") or main.get("headline")
    flat["description"] = main.get("description")
    flat["date_published"] = main.get("datePublished")
    flat["date_modified"] = main.get("dateModified")

    author = main.get("author")
    if isinstance(author, dict):
        flat["author"] = author.get("name")
    elif isinstance(author, list):
        flat["author"] = "; ".join(a.get("name", "") for a in author if isinstance(a, dict))
    else:
        flat["author"] = author

    flat["birth_date"] = main.get("birthDate")
    flat["death_date"] = main.get("deathDate")
    flat["keywords"] = main.get("keywords")

    flat["full_text"] = record.get("full_text") or ""
    flat["full_text_word_count"] = record.get("full_text_word_count") or 0

    return flat


def export_csv(jsonl_path, csv_path):
    by_url = {}
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            by_url[rec.get("_source_url")] = flatten_jsonld(rec)

    rows = list(by_url.values())

    if not rows:
        print(f"  (no rows to export for {jsonl_path.name})")
        return

    fieldnames = list(rows[0].keys())
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"  wrote {len(rows)} rows -> {csv_path}")


def main():
    parser = argparse.ArgumentParser(description="Build a dataset from sahistory.org.za")
    parser.add_argument(
        "--types", nargs="+",
        choices=["biography", "article", "archive", "place", "event"],
        default=["biography", "article", "archive", "place", "event"],
    )
    parser.add_argument("--limit", type=int, default=None)
    parser.add_argument("--out", type=Path, default=Path("./saho_dataset"))
    parser.add_argument("--delay", type=float, default=CRAWL_DELAY)
    parser.add_argument("--csv", action="store_true")
    args = parser.parse_args()

    args.out.mkdir(parents=True, exist_ok=True)
    session = requests.Session()

    page_types = [t for t in args.types if t != "event"]

    if page_types:
        print("Discovering URLs from sitemap.xml (this walks the full sitemap index)...")
        all_urls = get_all_sitemap_urls(session)
        print(f"Found {len(all_urls)} total URLs in sitemap")

        by_type = {t: [] for t in page_types}
        for url in all_urls:
            t = classify_url(url)
            if t in by_type:
                by_type[t].append(url)

        for t in page_types:
            print(f"[{t}] {len(by_type[t])} URLs matched")
            out_path = args.out / f"{t}.jsonl"
            scrape_page_type(t, by_type[t], out_path, args.limit, session, args.delay)

    if "event" in args.types:
        out_path = args.out / "event.jsonl"
        scrape_events_via_timeline_api(out_path, session, args.delay, args.limit)

    if args.csv:
        print("\nExporting CSVs...")
        for t in args.types:
            jsonl_path = args.out / f"{t}.jsonl"
            if jsonl_path.exists():
                export_csv(jsonl_path, args.out / f"{t}.csv")

    print("\nDone. Remember: SAHO content is CC BY-NC-SA 4.0 -- keep attribution,")
    print("non-commercial use only, and share any derivative dataset under the same license.")


if __name__ == "__main__":
    main()

Overwriting sahistory_scraper.py


Download only biography and article (50 records only )

In [16]:
with open("sahistory_scraper.py") as f:
    print("full_text logic present:", "extract_full_text" in f.read())

!python sahistory_scraper.py --types biography article --limit 50 --out /content/drive/MyDrive/saho_dataset --csv

full_text logic present: True
Discovering URLs from sitemap.xml (this walks the full sitemap index)...
Found 104963 total URLs in sitemap
[biography] 10748 URLs matched
[biography] resuming: 0 already saved with full text
[biography] fetching 50 pages...
biography: 100% 50/50 [00:59<00:00,  1.19s/it]
[article] 2813 URLs matched
[article] resuming: 0 already saved with full text
[article] fetching 50 pages...
article: 100% 50/50 [00:57<00:00,  1.16s/it]

Exporting CSVs...
  wrote 50 rows -> /content/drive/MyDrive/saho_dataset/biography.csv
  wrote 50 rows -> /content/drive/MyDrive/saho_dataset/article.csv

Done. Remember: SAHO content is CC BY-NC-SA 4.0 -- keep attribution,
non-commercial use only, and share any derivative dataset under the same license.


To get all five types ( 50 record each)

In [ ]:
!python sahistory_scraper.py --types biography article archive place event --out /content/drive/MyDrive/saho_dataset --limit 50 --csv

FULL FORCE!!! DOWNLOAD ALL CONTENT BEWARE LOL IT TAKES HOURS

In [ ]:
!python sahistory_scraper.py --types biography article archive place event --out /content/drive/MyDrive/saho_dataset --csv